# ValidEval lm-eval Panel Runner

Generates normalized open-model prediction panels for ValidEval import. No API keys or paid models are used.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

OUTPUT_ROOT = Path("/kaggle/working/valideval_panel_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("python", sys.version)
try:
    import torch

    print("cuda_available", torch.cuda.is_available())
    print("gpu_count", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("gpu_name", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch_check_failed", repr(exc))

In [ ]:
%pip install -q -r /kaggle/input/valideval-kaggle-package/kaggle/kaggle_requirements.txt

In [ ]:
import yaml

PACKAGE_ROOT = Path("/kaggle/input/valideval-kaggle-package/kaggle")
MODEL_CONFIG = PACKAGE_ROOT / "panel_models_small.yaml"
TASK_CONFIG = PACKAGE_ROOT / "panel_tasks.yaml"
models = yaml.safe_load(MODEL_CONFIG.read_text())["models"]
tasks_cfg = yaml.safe_load(TASK_CONFIG.read_text())
selected_task_ids = tasks_cfg["defaults"]["selected_tasks"]
tasks = [task for task in tasks_cfg["tasks"] if task["id"] in selected_task_ids]
limit = tasks_cfg["defaults"].get("limit")
print("models", [m["id"] for m in models])
print("tasks", [t["id"] for t in tasks], "limit", limit)

In [ ]:
def run_lm_eval(model_cfg, task_cfg):
    model_id = model_cfg["id"]
    task_name = task_cfg["lm_eval_task"]
    safe_model = model_id.replace("/", "__")
    run_dir = OUTPUT_ROOT / task_cfg["benchmark"] / safe_model
    done_file = run_dir / ".done"
    run_dir.mkdir(parents=True, exist_ok=True)
    if done_file.exists():
        print("resume_skip", model_id, task_name)
        return run_dir
    cmd = [
        sys.executable,
        "-m",
        "lm_eval",
        "--model",
        model_cfg.get("backend", "hf"),
        "--model_args",
        f"pretrained={model_id},dtype={model_cfg.get('dtype', 'auto')}",
        "--tasks",
        task_name,
        "--output_path",
        str(run_dir),
        "--log_samples",
    ]
    if model_cfg.get("batch_size"):
        cmd.extend(["--batch_size", str(model_cfg["batch_size"])])
    if limit:
        cmd.extend(["--limit", str(limit)])
    print("running", " ".join(cmd))
    subprocess.run(cmd, check=True)
    done_file.write_text(str(time.time()))
    return run_dir


run_dirs = []
for task_cfg in tasks:
    for model_cfg in models:
        run_dirs.append(run_lm_eval(model_cfg, task_cfg))

In [ ]:
import pandas as pd


def iter_sample_rows(run_dir):
    for path in Path(run_dir).rglob("samples_*.jsonl"):
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    yield path, json.loads(line)


def normalize_row(path, row, model_id, benchmark):
    doc = row.get("doc") or {}
    item_id = str(
        row.get("doc_id")
        or doc.get("id")
        or doc.get("question_id")
        or row.get("arguments")
        or hashlib.sha256(json.dumps(doc, sort_keys=True).encode()).hexdigest()[:16]
    )
    gold = row.get("target") if row.get("target") is not None else row.get("gold")
    prediction = row.get("filtered_resps") or row.get("resps") or row.get("prediction")
    exact = row.get("exact_match")
    if exact is None:
        metrics = row.get("metrics") or {}
        exact = metrics.get("exact_match") or metrics.get("acc")
    correct = bool(exact) if exact is not None else None
    return {
        "schema_version": "0.1",
        "benchmark": benchmark,
        "subset": row.get("task_name") or benchmark,
        "item_id": item_id,
        "model_id": model_id,
        "gold": gold,
        "prediction": prediction,
        "correct": correct,
        "source": "kaggle_lm_eval",
        "source_file": str(path.name),
        "metadata": {"lm_eval_sample": True},
    }


records = []
for task_cfg in tasks:
    for model_cfg in models:
        safe_model = model_cfg["id"].replace("/", "__")
        run_dir = OUTPUT_ROOT / task_cfg["benchmark"] / safe_model
        for path, row in iter_sample_rows(run_dir):
            records.append(normalize_row(path, row, model_cfg["id"], task_cfg["benchmark"]))

predictions_path = OUTPUT_ROOT / "predictions.jsonl"
with predictions_path.open("w", encoding="utf-8") as handle:
    for row in records:
        handle.write(json.dumps(row, sort_keys=True) + "\n")
print("normalized_rows", len(records), predictions_path)

In [ ]:
df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError("No normalized rows found. Check lm-eval sample outputs.")
if df["correct"].isna().any():
    raise RuntimeError(
        "Some rows have no deterministic correctness field; do not import until scorer is fixed."
    )
matrix = df.pivot_table(index="model_id", columns="item_id", values="correct", aggfunc="first")
matrix = matrix.astype(float).reset_index()
matrix_path = OUTPUT_ROOT / "matrix.csv"
matrix.to_csv(matrix_path, index=False)
print("matrix_shape", matrix.shape, matrix_path)

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "schema_version": "0.1",
    "created_at_unix": time.time(),
    "models": [m["id"] for m in models],
    "tasks": [t["id"] for t in tasks],
    "row_count": len(records),
    "matrix_shape": list(matrix.shape),
    "predictions_path": str(predictions_path),
    "matrix_path": str(matrix_path),
    "sha256": {
        "predictions_jsonl": file_sha256(predictions_path),
        "matrix_csv": file_sha256(matrix_path),
    },
    "evidence_state_until_local_import": "RESULT_REQUIRED",
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
zip_base = "/kaggle/working/valideval_kaggle_outputs"
shutil.make_archive(zip_base, "zip", OUTPUT_ROOT)
print(json.dumps(manifest, indent=2, sort_keys=True))
print("zip", zip_base + ".zip")